# Chronos forecasting POC

This notebook isolates each method from the original script into its own code cell so they can be executed independently or re-used in a notebook workflow.

The source file being mirrored is `poc copy.py`.


## 1) Source inspection

This first section reads the original script from disk and parses the Python module structure. It helps confirm which imports and top-level functions belong in the notebook before we isolate them into separate cells.


In [ ]:
from pathlib import Path
import ast

source_path = Path.cwd() / 'poc copy.py'
source_text = source_path.read_text(encoding='utf-8')
print(f'Loaded source: {source_path}')
print(f'Characters: {len(source_text)}')

module = ast.parse(source_text)
print(f'Top-level nodes: {len(module.body)}')


### Read the source file

This cell reads the active Python file and prints basic metadata. It is useful for verifying that the source file exists and has the expected structure before extracting functions.


In [ ]:
import ast

imports = []
for node in module.body:
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        imports.append(ast.unparse(node))

imports = list(dict.fromkeys(imports))
print('Normalized imports:')
for item in imports:
    print(item)
NORMALIZED_IMPORTS = imports


### Import extraction

This cell uses the Python AST to collect and normalize import statements. It removes duplicates and keeps the top-level dependencies together in one location before the method cells are executed.


In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

from chronos import BaseChronosPipeline

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"

AAPL_FILE = DATA_DIR / "apple_historical.csv"
SPX_FILE = DATA_DIR / "S&P 500 Historical Data.csv"
VIX_FILE = DATA_DIR / "VIXCLS.csv"

CHRONOS_MODEL = "amazon/chronos-2"
DEVICE = "cuda"

N_WINDOWS = 25
HORIZONS = [1, 5, 10, 20]
LOOKBACK = 252
CHRONOS_CONTEXT = 512
SPIKE_SIGMA = 2.0
QUANTILES = [0.05, 0.10, 0.50, 0.90, 0.95]

print('Imports and config loaded.')
print(f'ROOT={ROOT}')
print(f'DATA_DIR={DATA_DIR}')
print(f'RESULTS_DIR={RESULTS_DIR}')


### Shared configuration

This cell reproduces the project constants and model settings from the source script. These values are reused by the data-loading and forecasting functions in the later cells.


In [ ]:
def clean_number(value) -> float:
    """Convert common market-data strings to floats."""
    if pd.isna(value):
        return np.nan

    s = str(value).strip()
    s = s.replace("$", "")
    s = s.replace(",", "")
    s = s.replace("%", "")

    if s in {"", "-", "—", "nan", "NaN", "None"}:
        return np.nan

    try:
        return float(s)
    except ValueError:
        return np.nan


In [ ]:
### clean_number

This helper converts noisy market strings like "$229.98", "2,345", and "-" into valid numeric values. It is used by the CSV parsing functions to normalize historical price and volume data.


In [ ]:
def find_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    """Find a column case-insensitively from candidate names."""
    normalized = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]

    for candidate in candidates:
        key = candidate.strip().lower()
        for norm, original in normalized.items():
            if key in norm:
                return original

    return None


### find_column

This helper looks for a column name in a case-insensitive way using a list of likely candidates. It is important because market datasets often use slightly different column names such as `Close`, `Adj Close`, or `Close/Last`.


In [ ]:
def parse_market_file(path: Path, prefix: str) -> pd.DataFrame:
    """Load a market CSV and standardize Date + OHLCV where available."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}.\n"
            f"Expected file: {path.resolve()}"
        )

    df = pd.read_csv(path)

    date_col = find_column(
        df,
        ["Date", "date", "timestamp", "observation_date"]
    )

    if date_col is None:
        raise ValueError(
            f"Could not find a date column in {path.name}. "
            f"Columns found: {list(df.columns)}"
        )

    df["Date"] = pd.to_datetime(
        df[date_col],
        errors="coerce"
    )

    result = pd.DataFrame({"Date": df["Date"]})

    close_col = find_column(
        df,
        ["Close/Last", "Close", "Price", "Adj Close", "Last", "VIXCLS"]
    )

    if close_col is not None:
        result[f"{prefix}_Close"] = df[close_col].map(clean_number)

    for field, candidates in {
        "Open": ["Open"],
        "High": ["High"],
        "Low": ["Low"],
        "Volume": ["Volume"],
    }.items():
        col = find_column(df, candidates)
        if col is not None:
            result[f"{prefix}_{field}"] = df[col].map(clean_number)

    result = result.dropna(subset=["Date"])
    result = (
        result
        .sort_values("Date")
        .drop_duplicates("Date")
        .reset_index(drop=True)
    )

    return result


### parse_market_file

This function loads a single market CSV, detects the date column, standardizes the naming across files, and converts OHLCV values to numeric floats. It is the normalization step before the dataset can be merged and compared across markets.


In [ ]:
def load_and_align_data() -> pd.DataFrame:
    """Load AAPL, SPX and VIX and align them on AAPL trading dates."""

    aapl = parse_market_file(
        AAPL_FILE,
        "AAPL"
    )

    spx = parse_market_file(
        SPX_FILE,
        "SPX"
    )

    vix = parse_market_file(
        VIX_FILE,
        "VIX"
    )

    if "AAPL_Close" not in aapl.columns:
        raise ValueError(
            "Could not identify the Apple close column."
        )

    if "SPX_Close" not in spx.columns:
        raise ValueError(
            "Could not identify the S&P 500 close column."
        )

    if "VIX_Close" not in vix.columns:
        raise ValueError(
            "Could not identify the VIX close/value column."
        )

    df = aapl.merge(
        spx[["Date", "SPX_Close"]],
        on="Date",
        how="left"
    )

    df = df.merge(
        vix[["Date", "VIX_Close"]],
        on="Date",
        how="left"
    )

    df["SPX_Close"] = df["SPX_Close"].ffill()
    df["VIX_Close"] = df["VIX_Close"].ffill()

    df = df.dropna(subset=["AAPL_Close"])
    df = df.reset_index(drop=True)

    return df


### load_and_align_data

This is the master data-loading step. It combines the AAPL, SPX, and VIX datasets into one aligned time series so the experiments can use a common date index and a consistent set of features.


In [ ]:
def get_cutoffs(n_rows: int) -> list[int]:
    max_horizon = max(HORIZONS)

    start = LOOKBACK + 30
    end = n_rows - max_horizon - 1

    if end <= start:
        raise ValueError(
            f"Not enough data. Need more than {LOOKBACK + max_horizon + 30} rows."
        )

    candidates = np.arange(start, end + 1)

    if len(candidates) <= N_WINDOWS:
        return candidates.tolist()

    indices = np.linspace(
        0,
        len(candidates) - 1,
        N_WINDOWS,
        dtype=int
    )

    return candidates[indices].tolist()


### get_cutoffs

This method determines the walk-forward validation cutoffs. It ensures the model has enough history for the lookback window and reserves enough future data for each prediction horizon while sampling a limited number of windows for efficiency.


In [ ]:
def make_features(
    df: pd.DataFrame,
    cross_market: bool = False
) -> pd.DataFrame:
    """Features available at the cutoff date only."""

    x = df.copy()

    close = x["AAPL_Close"]

    x["ret_1"] = close.pct_change(1)
    x["ret_2"] = close.pct_change(2)
    x["ret_3"] = close.pct_change(3)
    x["ret_5"] = close.pct_change(5)
    x["ret_10"] = close.pct_change(10)
    x["ret_20"] = close.pct_change(20)

    if all(c in x.columns for c in [
        "AAPL_Open",
        "AAPL_High",
        "AAPL_Low"
    ]):
        x["intraday_range"] = (
            x["AAPL_High"] - x["AAPL_Low"]
        ) / close

        x["open_close_return"] = (
            x["AAPL_Close"] - x["AAPL_Open"]
        ) / x["AAPL_Open"]

    if "AAPL_Volume" in x.columns:
        x["volume_change"] = x["AAPL_Volume"].pct_change()
        x["volume_z20"] = (
            x["AAPL_Volume"]
            - x["AAPL_Volume"].rolling(20).mean()
        ) / (
            x["AAPL_Volume"].rolling(20).std()
            + 1e-8
        )

    x["vol_5"] = x["ret_1"].rolling(5).std()
    x["vol_20"] = x["ret_1"].rolling(20).std()
    x["vol_60"] = x["ret_1"].rolling(60).std()

    x["mom_5"] = close / close.shift(5) - 1
    x["mom_20"] = close / close.shift(20) - 1
    x["mom_60"] = close / close.shift(60) - 1

    if cross_market:
        for col, name in [
            ("SPX_Close", "spx"),
            ("VIX_Close", "vix"),
        ]:
            if col in x.columns:
                x[f"{name}_ret_1"] = x[col].pct_change(1)
                x[f"{name}_ret_5"] = x[col].pct_change(5)
                x[f"{name}_ret_20"] = x[col].pct_change(20)

                x[f"{name}_level_z20"] = (
                    x[col]
                    - x[col].rolling(20).mean()
                ) / (
                    x[col].rolling(20).std()
                    + 1e-8
                )

    return x


### make_features

This function creates all the statistical features used by the XGBoost baseline: returns, momentum, volatility, intraday range, volume change, and optional cross-market features. These features represent only information available at the forecast cutoff.


In [ ]:
def get_feature_columns(
    data: pd.DataFrame,
    cross_market: bool = False
) -> list[str]:

    preferred = [
        "ret_1",
        "ret_2",
        "ret_3",
        "ret_5",
        "ret_10",
        "ret_20",
        "intraday_range",
        "open_close_return",
        "volume_change",
        "volume_z20",
        "vol_5",
        "vol_20",
        "vol_60",
        "mom_5",
        "mom_20",
        "mom_60",
    ]

    if cross_market:
        preferred += [
            "spx_ret_1",
            "spx_ret_5",
            "spx_ret_20",
            "spx_level_z20",
            "vix_ret_1",
            "vix_ret_5",
            "vix_ret_20",
            "vix_level_z20",
        ]

    return [c for c in preferred if c in data.columns]


### get_feature_columns

This helper selects the subset of engineered features that are actually present in the data frame. It keeps the model input stable even when some optional columns are unavailable for a given experiment.


In [ ]:
def xgb_predict_endpoint(
    history: pd.DataFrame,
    horizon: int,
    cross_market: bool = False,
) -> float:
    """
    Direct multi-horizon regression:
    predict the return from T -> T+h.

    This avoids using unknown future OHLCV/covariates recursively.
    """

    work = make_features(
        history,
        cross_market=cross_market
    )

    work["target"] = (
        work["AAPL_Close"].shift(-horizon)
        / work["AAPL_Close"]
        - 1
    )

    features = get_feature_columns(
        work,
        cross_market=cross_market
    )

    train = work.dropna(
        subset=features + ["target"]
    ).tail(LOOKBACK)

    if len(train) < 100:
        raise ValueError(
            "Not enough valid training rows for XGBoost."
        )

    model = XGBRegressor(
        n_estimators=350,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        train[features],
        train["target"]
    )

    latest = work.iloc[-1]
    X_latest = latest[features].values.reshape(1, -1)

    predicted_return = float(
        model.predict(X_latest)[0]
    )

    last_price = float(
        history["AAPL_Close"].iloc[-1]
    )

    return last_price * (1 + predicted_return)


### xgb_predict_endpoint

This function trains a direct-horizon XGBoost regressor and predicts the endpoint price for one forecasting horizon. It uses features only from the historical data known at the cutoff date, which matches the walk-forward evaluation setup.


In [ ]:
def build_chronos_context(
    history: pd.DataFrame,
    experiment: int,
) -> pd.DataFrame:
    """
    Build the pandas input expected by Chronos-2.

    Exp 1: target = AAPL_Close only
    Exp 2: target = AAPL_Close + AAPL OHLCV as past covariates
    Exp 3: target = AAPL_Close + SPX/VIX as past covariates
    """

    context = pd.DataFrame({
        "id": "AAPL",
        "timestamp": history["Date"].values,
        "target": history["AAPL_Close"].values,
    })

    if experiment == 2:
        for source, name in [
            ("AAPL_Open", "open"),
            ("AAPL_High", "high"),
            ("AAPL_Low", "low"),
            ("AAPL_Volume", "volume"),
        ]:
            if source in history.columns:
                context[name] = history[source].values

    elif experiment == 3:
        for source, name in [
            ("SPX_Close", "spx"),
            ("VIX_Close", "vix"),
        ]:
            if source in history.columns:
                context[name] = history[source].values

    return context.tail(CHRONOS_CONTEXT).copy()


### build_chronos_context

This routine converts the historical data into the pandas format expected by Chronos-2. Depending on the experiment, it includes univariate prices, OHLCV covariates, or cross-market covariates like SPX and VIX.


In [ ]:
def chronos_predict_endpoint(
    pipeline: BaseChronosPipeline,
    history: pd.DataFrame,
    horizon: int,
    experiment: int,
) -> dict[str, float]:

    context = build_chronos_context(
        history,
        experiment=experiment
    )

    pred = pipeline.predict_df(
        context,
        prediction_length=horizon,
        quantile_levels=QUANTILES,
        id_column="id",
        timestamp_column="timestamp",
        target="target",
    )

    return {
        "p05": float(pred["0.05"].iloc[-1]),
        "p10": float(pred["0.1"].iloc[-1]),
        "p50": float(pred["0.5"].iloc[-1]),
        "p90": float(pred["0.9"].iloc[-1]),
        "p95": float(pred["0.95"].iloc[-1]),
    }


### chronos_predict_endpoint

This cell calls the Chronos-2 model to produce a forecast distribution for the requested horizon. It returns the key quantile predictions, especially P10, P50, and P90, which are later compared with the actual future price.


In [ ]:
def run_experiment(
    df: pd.DataFrame,
    pipeline: BaseChronosPipeline,
    experiment: int,
) -> pd.DataFrame:

    names = {
        1: "UNIVARIATE",
        2: "OHLCV",
        3: "CROSS_MARKET",
    }

    print("\n" + "=" * 80)
    print(f"EXPERIMENT {experiment} — {names[experiment]}")
    print("=" * 80)

    cross_market = experiment == 3
    cutoffs = get_cutoffs(len(df))
    rows = []

    for window_i, cutoff in enumerate(cutoffs, start=1):
        print(
            f"  window {window_i}/{len(cutoffs)} "
            f"(cutoff={df['Date'].iloc[cutoff - 1].date()})"
        )

        history = df.iloc[:cutoff].copy()

        for horizon in HORIZONS:
            future = df.iloc[cutoff: cutoff + horizon].copy()
            actual_final = float(future["AAPL_Close"].iloc[-1])
            last_price = float(history["AAPL_Close"].iloc[-1])

            ml_price = xgb_predict_endpoint(
                history,
                horizon=horizon,
                cross_market=cross_market,
            )

            chronos = chronos_predict_endpoint(
                pipeline,
                history,
                horizon=horizon,
                experiment=experiment,
            )

            rows.append({
                "experiment": names[experiment],
                "cutoff_date": history["Date"].iloc[-1],
                "horizon": horizon,
                "last_price": last_price,
                "actual_price": actual_final,
                "xgb_price": ml_price,
                "xgb_abs_error": abs(ml_price - actual_final),
                "chronos_p05": chronos["p05"],
                "chronos_p10": chronos["p10"],
                "chronos_p50": chronos["p50"],
                "chronos_p90": chronos["p90"],
                "chronos_p95": chronos["p95"],
                "chronos_abs_error": abs(chronos["p50"] - actual_final),
                "actual_return": actual_final / last_price - 1,
                "xgb_return": ml_price / last_price - 1,
                "chronos_p10_return": chronos["p10"] / last_price - 1,
                "chronos_p50_return": chronos["p50"] / last_price - 1,
                "chronos_p90_return": chronos["p90"] / last_price - 1,
            })

    result = pd.DataFrame(rows)

    filename = {
        1: "experiment_1_univariate.csv",
        2: "experiment_2_ohlcv.csv",
        3: "experiment_3_cross_market.csv",
    }[experiment]

    result.to_csv(RESULTS_DIR / filename, index=False)
    return result


### run_experiment

This is the main experimental loop. It selects walk-forward cutoffs, predicts every horizon using both models, and stores the results in a DataFrame for later evaluation.


In [ ]:
def run_spike_experiment(
    df: pd.DataFrame,
    pipeline: BaseChronosPipeline,
) -> pd.DataFrame:
    """One-day-ahead tail-event test."""

    print("\n" + "=" * 80)
    print("SPIKE EXPERIMENT — 1 DAY AHEAD")
    print("=" * 80)

    rows = []
    cutoffs = get_cutoffs(len(df))

    for i, cutoff in enumerate(cutoffs, start=1):
        history = df.iloc[:cutoff].copy()
        future = df.iloc[cutoff]

        last_price = float(history["AAPL_Close"].iloc[-1])
        actual_return = float(future["AAPL_Close"]) / last_price - 1
        returns = history["AAPL_Close"].pct_change().dropna()
        recent_vol = float(returns.tail(20).std())
        threshold = SPIKE_SIGMA * recent_vol
        actual_spike = abs(actual_return) > threshold

        chronos = chronos_predict_endpoint(
            pipeline,
            history,
            horizon=1,
            experiment=3,
        )

        chronos_p10_return = chronos["p10"] / last_price - 1
        chronos_p90_return = chronos["p90"] / last_price - 1
        chronos_warning = (
            chronos_p10_return < -threshold or chronos_p90_return > threshold
        )

        xgb_price = xgb_predict_endpoint(history, horizon=1, cross_market=True)
        xgb_return = xgb_price / last_price - 1
        xgb_warning = abs(xgb_return) > threshold

        rows.append({
            "cutoff_date": history["Date"].iloc[-1],
            "actual_return": actual_return,
            "recent_vol": recent_vol,
            "spike_threshold": threshold,
            "actual_spike": actual_spike,
            "chronos_p10_return": chronos_p10_return,
            "chronos_p50_return": chronos["p50"] / last_price - 1,
            "chronos_p90_return": chronos_p90_return,
            "chronos_warning": chronos_warning,
            "xgb_return": xgb_return,
            "xgb_warning": xgb_warning,
        })

        if i % 5 == 0 or i == len(cutoffs):
            print(f"  processed {i}/{len(cutoffs)}")

    result = pd.DataFrame(rows)
    result.to_csv(RESULTS_DIR / "spike_results.csv", index=False)
    return result


### run_spike_experiment

This function evaluates a one-day-ahead spike detection setup. It compares the forecasted return quantile range from Chronos-2 with the point forecast from XGBoost against a threshold based on recent volatility.


In [ ]:
def summarize_forecasts(result: pd.DataFrame) -> pd.DataFrame:
    summary = (
        result
        .groupby("horizon")
        .agg(
            xgb_mae=("xgb_abs_error", "mean"),
            chronos_mae=("chronos_abs_error", "mean"),
        )
        .reset_index()
    )

    summary["chronos_mae_improvement_pct"] = (
        (summary["xgb_mae"] - summary["chronos_mae"]) / summary["xgb_mae"] * 100
    )

    return summary


### summarize_forecasts

This summary aggregates absolute errors by forecast horizon so we can compare the average performance of XGBoost and Chronos-2 across 1-, 5-, 10-, and 20-day predictions.


In [ ]:
def binary_metrics(
    actual: pd.Series,
    predicted: pd.Series,
) -> tuple[float, float]:

    actual = actual.astype(bool).values
    predicted = predicted.astype(bool).values

    tp = np.sum(actual & predicted)
    fp = np.sum(~actual & predicted)
    fn = np.sum(actual & ~predicted)

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0

    return precision, recall


### binary_metrics

This helper computes precision and recall for binary classification-style event detection. It is used in the spike experiment to evaluate whether the model flags actual extremes without too many false alarms.


In [ ]:
def summarize_spikes(spike_df: pd.DataFrame) -> pd.DataFrame:
    chronos_precision, chronos_recall = binary_metrics(
        spike_df["actual_spike"],
        spike_df["chronos_warning"],
    )

    xgb_precision, xgb_recall = binary_metrics(
        spike_df["actual_spike"],
        spike_df["xgb_warning"],
    )

    summary = pd.DataFrame([
        {"model": "Chronos-2", "precision": chronos_precision, "recall": chronos_recall},
        {"model": "XGBoost", "precision": xgb_precision, "recall": xgb_recall},
    ])

    return summary


### summarize_spikes

This table combines the precision and recall results for Chronos-2 and XGBoost on the spike experiment. It tells us which model better identifies unusual directional moves.


In [ ]:
def summarize_calibration(
    result: pd.DataFrame,
) -> pd.DataFrame:
    """Check whether Chronos P10/P90 behave like their nominal tails."""

    rows = []

    for horizon, g in result.groupby("horizon"):
        last = g["last_price"]
        actual_ret = g["actual_return"]
        p10_ret = g["chronos_p10"] / last - 1
        p90_ret = g["chronos_p90"] / last - 1

        rows.append({
            "horizon": horizon,
            "p10_breach_rate": float(np.mean(actual_ret < p10_ret)),
            "p90_breach_rate": float(np.mean(actual_ret > p90_ret)),
            "expected_p10_breach_rate": 0.10,
            "expected_p90_breach_rate": 0.10,
        })

    return pd.DataFrame(rows)


### summarize_calibration

This method checks whether the Chronos quantile intervals behave like calibrated prediction intervals. It compares the realized breach rates of the P10 and P90 quantiles with the expected nominal rates.


In [ ]:
def plot_last_forecast(
    df: pd.DataFrame,
    pipeline: BaseChronosPipeline,
    experiment: int = 3,
    horizon: int = 20,
) -> None:

    cutoff = len(df) - horizon
    history = df.iloc[:cutoff].copy()
    future = df.iloc[cutoff:cutoff + horizon].copy()

    chronos_context = build_chronos_context(history, experiment=experiment)
    pred = pipeline.predict_df(
        chronos_context,
        prediction_length=horizon,
        quantile_levels=QUANTILES,
        id_column="id",
        timestamp_column="timestamp",
        target="target",
    )

    xgb_endpoint = xgb_predict_endpoint(
        history,
        horizon=horizon,
        cross_market=(experiment == 3),
    )

    p10 = pred["0.1"].values
    p50 = pred["0.5"].values
    p90 = pred["0.9"].values

    plt.figure(figsize=(14, 7))
    plt.plot(history["Date"].tail(100), history["AAPL_Close"].tail(100), label="Known AAPL history")
    plt.plot(future["Date"], future["AAPL_Close"], linewidth=3, label="Actual future")
    plt.plot(future["Date"], p50, linestyle="--", label="Chronos-2 P50")
    plt.fill_between(future["Date"], p10, p90, alpha=0.2, label="Chronos-2 P10-P90")
    plt.scatter([future["Date"].iloc[-1]], [xgb_endpoint], marker="x", s=90, label="XGBoost endpoint")
    plt.axvline(history["Date"].iloc[-1], linestyle=":", label="Forecast start")
    plt.title("AAPL — Chronos-2 vs Statistical ML")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "last_forecast.png", dpi=150)
    plt.show()


### plot_last_forecast

This produces the final comparison chart: actual future price, Chronos prediction interval, and XGBoost endpoint forecast. It visualizes how the two models differ at a single horizon.


In [ ]:
def main() -> None:

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("CHRONOS-2 PROXY / FORECASTING POC")
    print("=" * 80)

    print("\nLoading data...")
    df = load_and_align_data()
    print(f"AAPL rows after alignment: {len(df)}")
    print(f"Date range: {df['Date'].min().date()} -> {df['Date'].max().date()}")
    print("\nColumns:")
    print(list(df.columns))

    print("\nLoading Chronos-2...")
    pipeline = BaseChronosPipeline.from_pretrained(
        CHRONOS_MODEL,
        device_map=DEVICE,
    )
    print("Chronos-2 loaded.")

    all_summaries = []
    for experiment in [1, 2, 3]:
        result = run_experiment(df, pipeline, experiment=experiment)
        summary = summarize_forecasts(result)
        summary["experiment"] = experiment
        all_summaries.append(summary)
        print("\nSummary:")
        print(summary.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
        calibration = summarize_calibration(result)
        calibration.to_csv(RESULTS_DIR / f"experiment_{experiment}_calibration.csv", index=False)

    combined = pd.concat(all_summaries, ignore_index=True)
    combined.to_csv(RESULTS_DIR / "all_experiment_summary.csv", index=False)

    spikes = run_spike_experiment(df, pipeline)
    spike_summary = summarize_spikes(spikes)
    spike_summary.to_csv(RESULTS_DIR / "spike_summary.csv", index=False)

    print("\n" + "=" * 80)
    print("SPIKE SUMMARY")
    print("=" * 80)
    print(spike_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    print("\nCreating final visualization...")
    plot_last_forecast(df, pipeline, experiment=3, horizon=20)

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)
    print(f"Results are in: {RESULTS_DIR.resolve()}")


if __name__ == "__main__":
    main()


### main

This final orchestration cell runs the full experiment pipeline: load data, train the models, evaluate each experiment, save summary CSVs, run the spike test, and generate the final chart. It is the entry point for the entire data-science workflow.
